In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
import time
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values

print("CWD:", os.getcwd())                     # 현재 작업 경로 확인
print(".env path:", find_dotenv())             # 찾은 .env 경로 확인

# 기존 환경변수 덮어쓰기 허용!
load_dotenv(dotenv_path="env1.env", override=True)

# 값 읽기
SEARCH = os.getenv("SEARCH")
CONTRACT_DAY = os.getenv("CONTRACT_DAY")
APPLICATION_TYPE = os.getenv("APPLICATION_TYPE")
APP_GUBUN = os.getenv("APP_GUBUN")
NAME = os.getenv("NAME")
B_NUM=os.getenv("B_NUM")
S_NUM=os.getenv("S_NUM")
ORG_NAME= os.getenv("ORG_NAME")
BIRTH_DATE = os.getenv("BIRTH_DATE")
GENDER = os.getenv("GENDER")
CAR_MODEL = os.getenv("CAR_MODEL")
CAR_COUNT = int(os.getenv("CAR_COUNT", "0"))
DELIVERY_DATE = os.getenv("DELIVERY_DATE")
ADDRESS = os.getenv("ADDRESS")
ADDRESS2 = os.getenv("ADDRESS2")
PHONE = os.getenv("PHONE")
MOBILE = os.getenv("MOBILE")
EMAIL = os.getenv("EMAIL")
CONTACT_NAME = os.getenv("CONTACT_NAME")
CONTACT_MOBILE = os.getenv("CONTACT_MOBILE")
AGENCY_PHONE = os.getenv("AGENCY_PHONE")
MANUFACTURER_ID = os.getenv("MANUFACTURER_ID")
print(CONTRACT_DAY, APPLICATION_TYPE, NAME, CAR_MODEL)
# chromedriver.exe 경로 지정
# chrome_path = "C:\jupyter\chromedriver.exe"

print("시작")
url = "https://ev.or.kr/ev_ps/ps/seller/sellerApplyform?car_type=11"

# 브라우저 옵션
opts = Options()
# opts.add_argument("--headless=new")  # 필요시 헤드리스

# opts = Options()
opts.add_argument("--disable-popup-blocking")  # 팝업 차단 해제(중요)
opts.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

#driver = webdriver.Chrome(service=Service(chrome_path), options=opts)
#driver = webdriver.Chrome(options=opts)

# 크롬 드라이버 실행
# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
driver = webdriver.Chrome(options=opts)
# driver.get(url) #그페이지에서시작하도록

wait = WebDriverWait(driver, 10)

print("값 입력 시작!~~~~~~")
# 신청유형
select = Select(driver.find_element(By.ID, "req_kind"))
select.select_by_visible_text(APPLICATION_TYPE)

time.sleep(0.5)



# 계약일자 (readonly라 JS로 값 세팅)
driver.execute_script(
    f"document.getElementById('contract_day').value = '{CONTRACT_DAY}';"
)

#차종
select2 = Select(driver.find_element(By.ID, "model_cd"))
select2.select_by_visible_text(CAR_MODEL)

#신청대수
count_input = wait.until(EC.element_to_be_clickable((By.ID, "req_cnt")))  
count_input.send_keys(str(CAR_COUNT))

#출고예정일자
print("출고예정일 "+ DELIVERY_DATE)
driver.execute_script(
    f"document.getElementById('delivery_sch_day').value = '{DELIVERY_DATE}';"
)

#PHONE
phone_input = wait.until(EC.element_to_be_clickable((By.ID, "phone")))  
phone_input.send_keys(PHONE)

#휴대폰
mobile_input = wait.until(EC.element_to_be_clickable((By.ID, "mobile")))  
mobile_input.send_keys(MOBILE)

#이메일
email_input = wait.until(EC.element_to_be_clickable((By.ID, "email")))  
email_input.send_keys(EMAIL)

# 기관명 입력
name_input = wait.until(EC.element_to_be_clickable((By.ID, "req_nm")))  
name_input.send_keys(ORG_NAME)

#신청구분 grp_reqst_se
select2 = Select(driver.find_element(By.ID, "grp_reqst_se"))
select2.select_by_visible_text(APP_GUBUN)

#대표자 ceo
ceo_input = wait.until(EC.element_to_be_clickable((By.ID, "ceo")))  
ceo_input.send_keys(NAME)

#법인번호 birth2
b_input = wait.until(EC.element_to_be_clickable((By.ID, "birth2")))  
b_input.send_keys(B_NUM)


#사업자번호 busi_no
s_input = wait.until(EC.element_to_be_clickable((By.ID, "busi_no")))  
s_input.send_keys(S_NUM)


#개인사업장명 pri_busi_nm
nn_input = wait.until(EC.element_to_be_clickable((By.ID, "pri_busi_nm")))  
nn_input.send_keys(ORG_NAME)

seller_phone_input = wait.until(EC.element_to_be_clickable((By.ID, "seller_phone")))  
seller_phone_input.send_keys(AGENCY_PHONE)

contact_nm_input = wait.until(EC.element_to_be_clickable((By.ID, "contact_nm")))  
contact_nm_input.send_keys(CONTACT_NAME)

contact_mobile_input = wait.until(EC.element_to_be_clickable((By.ID, "contact_mobile")))  
contact_mobile_input.send_keys(CONTACT_MOBILE)

seller_mgrid_input = wait.until(EC.element_to_be_clickable((By.ID, "seller_mgrid")))  
seller_mgrid_input.send_keys(MANUFACTURER_ID)

print("주소")
# 1) 팝업 여는 버튼 클릭
parent2 = driver.current_window_handle
before2 = set(driver.window_handles)

open_btn = wait.until(EC.element_to_be_clickable((
    By.XPATH,
    "//*[@onclick and contains(@onclick, \"/ev_ps/addrlink/addrPopup\")]"
)))
open_btn.click()

# 2) 새 창(탭) 뜰 때까지 대기 후 전환
wait.until(lambda d: len(d.window_handles) > len(before2))
child2 = (set(driver.window_handles) - before2).pop()
driver.switch_to.window(child2)

# 3) 검색어 입력
KEYWORD = ADDRESS # 필요한 값으로 교체
kw = wait.until(EC.element_to_be_clickable((By.ID, "keyword")))
kw.clear()
kw.send_keys(KEYWORD)

# 4) 검색 버튼 클릭 (searchUrlJuso)
search_btn = wait.until(EC.element_to_be_clickable((
    By.XPATH,
    "//button[contains(@onclick, 'searchUrlJuso')]"
)))
search_btn.click()

# 1) 요소 등장 대기 (href 기준)
wait.until(lambda d: len(d.find_elements(By.XPATH, "//a[contains(@href, \"setMaping('1')\")]")) > 0)

# 2) 첫 번째 요소 잡아서 스크롤→JS 클릭 (오버레이/좌표 이슈 회피)
attempts = 3
for _ in range(attempts):
    try:
        a = driver.find_element(By.XPATH, "//a[contains(@href, \"setMaping('1')\")]")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", a)
        driver.execute_script("arguments[0].click();", a)
        break
    except StaleElementReferenceException:
        time.sleep(0.3)
else:
    raise RuntimeError("setMaping('1') 클릭 실패")

time.sleep(0.1)
adr2_input = wait.until(EC.element_to_be_clickable((By.ID, "rtAddrDetail")))  
adr2_input.send_keys(ADDRESS2)

btn = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//button[contains(@onclick, 'setParent')]"
)))
btn.click()


# 6) 필요하면 팝업 닫고 원래 창으로 복귀
# driver.close()
driver.switch_to.window(parent2)

btn = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//button[contains(@onclick, 'goSave')]"
)))
btn.click()
time.sleep(0.5)
# confirm 창으로 전환
#alert = driver.switch_to.alert

try:
    alert = wait.until(EC.alert_is_present())
    txt = alert.text  # 필요하면 메시지 확인
    alert.accept()    # 확인
    # alert.dismiss() # 취소
except TimeoutException:
    print("alert가 안 떴습니다.")



# ==== confirm OK 후 새창으로 전환해 작업 ====
parent3 = driver.current_window_handle
before3 = set(driver.window_handles)

# 이미 alert.accept()까지 한 상태라면, 여기서 다시 기다리지 않습니다.
# (혹시 타이밍상 아직 안눌렀다면 아래 주석 해제)
# WebDriverWait(driver, 5).until(EC.alert_is_present()).accept()

# 새창/탭이 열릴 때까지 대기 (핸들 개수 증가 기준)
WebDriverWait(driver, 10).until(lambda d: len(d.window_handles) > len(before3))

# 새로 열린 창 식별 후 전환
child3 = (set(driver.window_handles) - before3).pop()
driver.switch_to.window(child3)

# 로딩 완료 대기
wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

# 5) === 여기서 새창에서 필요한 작업 수행 ===
# 1) span.guide 안의 텍스트 가져오기
code_text = driver.find_element(
    By.XPATH, "//tbody/tr[2]/td[2]//span[@class='guide']"
).text.strip()
print("원본:", code_text)  # HU3cBT5NNS

# 2) 문자열 거꾸로 뒤집기
reversed_code = code_text[::-1]
print("뒤집은 값:", reversed_code)  # SNN5TBc3UH

# 3) input#randeomChk 에 입력
random_box = wait.until(EC.element_to_be_clickable((By.ID, "randeomChk")))
random_box.clear()
random_box.send_keys(reversed_code)

# 3) "확인" 버튼 클릭
confirm_btn = wait.until(EC.element_to_be_clickable((
    By.XPATH, "//button[contains(@onclick, 'goCompare')]"
)))
confirm_btn.click()

# 6) 작업 후 필요시 새창 닫고 원래 창으로 복귀
# driver.close()
driver.switch_to.window(parent3)

print("끝")


CWD: C:\Users\ssson
.env path: 
2025-09-01 단체 이정운 더뉴아이오닉5 AWD 롱레인지 19인치 / 일반승용
시작
값 입력 시작!~~~~~~
출고예정일 2025-09-12
주소
원본: d6HXE9F653
뒤집은 값: 356F9EXH6d
끝


In [2]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time, os
from selenium.common.exceptions import (
    ElementNotInteractableException, InvalidArgumentException,
    TimeoutException, NoAlertPresentException, NoSuchWindowException
)
from dotenv import load_dotenv

print("START!!!!")
load_dotenv("env1.env", override=True)
FILE1 = os.getenv("FILE1")

def accept_any_alert(driver, timeout=3, retries=2):
    handled = False
    for _ in range(retries):
        try:
            WebDriverWait(driver, timeout).until(EC.alert_is_present())
            a = driver.switch_to.alert
            txt = a.text
            time.sleep(0.1)
            a.accept()
            print("[alert] accepted:", txt)
            handled = True
            time.sleep(0.1)
        except (TimeoutException, NoAlertPresentException):
            break
        except NoSuchWindowException as e:
            print("[alert] 창 닫힘 감지 → 스킵:", e)
            handled = True
            break
        except Exception as e:
            print("[alert] 예외, 스킵:", repr(e))
            break
    return handled

def handle_attach(driver, wait, attach_id, file_path):
    """첨부 버튼 클릭 -> 팝업 전환 -> 파일 업로드 -> 저장 -> alert 확인 -> 부모 복귀"""
    # 매 호출 시점의 부모 핸들을 '현재 창'으로 잡음
    parent_handle = driver.current_window_handle
    before = set(driver.window_handles)

    # 1) 부모창에서 첨부 버튼 클릭
    attach_btn = wait.until(EC.element_to_be_clickable((
        By.XPATH, f"//button[contains(@onclick, \"popupAttachFile('{attach_id}')\")]"
    )))
    attach_btn.click()

    # 2) 새창 전환
    wait.until(lambda d: len(d.window_handles) > len(before))
    child = (set(driver.window_handles) - before).pop()
    driver.switch_to.window(child)
    wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

    # 3) 파일 input 찾기
    file_el = wait.until(EC.presence_of_element_located((By.ID, "filename")))

    # 3-1) 숨김/비활성 대비
    driver.execute_script("""
      const el = arguments[0];
      el.removeAttribute('disabled');
      el.removeAttribute('readonly');
      el.style.display = 'block';
      el.style.visibility = 'visible';
      el.style.opacity = 1;
      el.style.position = 'static';
      el.style.width = '420px';
      el.style.height = '32px';
      el.style.zIndex = 999999;
    """, file_el)

    time.sleep(0.2)
    # 4) 파일 경로 입력(재시도)
    last_err = None
    for attempt in range(2):
        try:
            file_el.send_keys(file_path)
            print(f"[{attach_id}] 파일 업로드 성공:", file_path)
            break
        except (ElementNotInteractableException, InvalidArgumentException, Exception) as e:
            last_err = e
            print(f"[{attach_id}] send_keys 시도 {attempt+1} 실패:", repr(e))
            time.sleep(0.3)
    else:
        raise RuntimeError(f"[{attach_id}] 파일 업로드 실패: {repr(last_err)}")

    # 5) 저장 버튼 클릭
    popup_form = wait.until(EC.presence_of_element_located((
        By.XPATH,
        "//form[@name='frm' and @action='/ev_ps/ps/comm/popupAttach/cud'"
        " and .//h1[contains(normalize-space(.), '첨부파일')]]"
    )))
    save_btn = popup_form.find_element(
        By.XPATH, ".//div[contains(@class,'content-button-group')]//button[contains(@class,'btn-blue') and contains(@onclick,'goSave')]"
    )
    wait.until(EC.element_to_be_clickable(save_btn))
    save_btn.click()

    # 6) 자식창에서 alert 먼저 처리
    handled_child = accept_any_alert(driver, timeout=5, retries=2)

    # 7) 자식창 닫힘/변화 대기
    child_handle = driver.current_window_handle
    try:
        WebDriverWait(driver, 3).until(lambda d: child_handle not in d.window_handles)
    except TimeoutException:
        # 닫히지 않으면 필요시 직접 닫아도 됨 (선택)
        # driver.close()
        pass

    # 8) 살아있는 창(가능하면 부모)으로 복귀
    try:
        if parent_handle in driver.window_handles:
            driver.switch_to.window(parent_handle)
        else:
            driver.switch_to.window(driver.window_handles[0])
    except NoSuchWindowException:
        # 그래도 문제면 남은 창 중 하나로
        driver.switch_to.window(driver.window_handles[0])

    # 9) 부모에서 alert 추가 처리
    handled_parent = accept_any_alert(driver, timeout=3, retries=2)

    print(f"[{attach_id}] 저장 처리 완료 (child_alert={handled_child}, parent_alert={handled_parent})")

# =================================================================

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
opts = Options()
opts.add_experimental_option("debuggerAddress", "127.0.0.1:9222")

driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, 10)

# === A1, A2, A3 모두 실행 ===
for attach_id in ["A", "A2", "A3"]:
    print("=== 처리 시작:", attach_id, "===")
    try:
        handle_attach(driver, wait, attach_id, FILE1)
    except Exception as e:
        # 한 건 실패해도 다음 건 계속
        print(f"[{attach_id}] 처리 실패 → 스킵: {repr(e)}")
        # 실패 후에도 부모쪽으로 복귀를 시도해서 다음 루프를 안정화
        try:
            if driver.window_handles:
                driver.switch_to.window(driver.window_handles[0])
        except Exception:
            pass
        continue

print("모든 첨부 완료(성공/스킵 포함) ✅")

# === 마지막 지원신청 버튼 클릭 ===
try:
    apply_btn = wait.until(EC.element_to_be_clickable((
        By.XPATH, "//button[contains(@onclick, \"goApply('101'\")]"
    )))
    apply_btn.click()
    print("[지원신청] 버튼 클릭 완료")

    # 지원신청시 alert 확인 (있을 가능성 높음)
    handled_apply = accept_any_alert(driver, timeout=5, retries=2)
    print("[지원신청] alert 처리:", handled_apply)

except Exception as e:
    print("[지원신청] 버튼 클릭 실패:", repr(e))

print("끝")


START!!!!
=== 처리 시작: A ===
[A] 파일 업로드 성공: C:\jupyter\file\영주이앤씨지원신청서.pdf
[alert] accepted: 등록 완료
[alert] 창 닫힘 감지 → 스킵: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=139.0.7258.155)
Stacktrace:
	GetHandleVerifier [0x0x7ff7ec003d85+79397]
	GetHandleVerifier [0x0x7ff7ec003de0+79488]
	(No symbol) [0x0x7ff7ebdac0fa]
	(No symbol) [0x0x7ff7ebd84601]
	(No symbol) [0x0x7ff7ebe3257e]
	(No symbol) [0x0x7ff7ebdd3cd6]
	(No symbol) [0x0x7ff7ebe2af73]
	(No symbol) [0x0x7ff7ebdf41b1]
	(No symbol) [0x0x7ff7ebdf4f43]
	GetHandleVerifier [0x0x7ff7ec2ce1ed+3005069]
	GetHandleVerifier [0x0x7ff7ec2c831d+2980797]
	GetHandleVerifier [0x0x7ff7ec2e7e0d+3110573]
	GetHandleVerifier [0x0x7ff7ec01d6de+184190]
	GetHandleVerifier [0x0x7ff7ec02516f+215567]
	GetHandleVerifier [0x0x7ff7ec00c974+115220]
	GetHandleVerifier [0x0x7ff7ec00cb29+115657]
	GetHandleVerifier [0x0x7ff7ebff3268+11016]
	BaseThreadInitThunk [0x0x7fffe87fe8d7+23]
	RtlUserThreadStart


KeyboardInterrupt

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001FD10F10050>>
Traceback (most recent call last):
  File "C:\Users\ssson\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel\ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
  File "C:\Users\ssson\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1479, in enumerate
    def enumerate():
KeyboardInterrupt: 


KeyboardInterrupt: 